# ClaimSense decision model v2: QLoRA fine-tuning (dataset v2, prompt decision_v3)

Accelerator: GPU T4 (the script uses one GPU). Internet: on. Input: the private `claimsense-decision-v2` dataset.
Run the smoke test first (`SMOKE = True`), then the full run (`SMOKE = False`). Outputs: `adapter/`, `gguf/`, `training-log.json`, `lengths.json`, `config.json`, `pip-freeze.txt`.

In [ ]:
!nvidia-smi
!pip install --upgrade --quiet unsloth unsloth_zoo

In [ ]:
import glob, os

# Finds the attached private dataset wherever Kaggle mounts it (any name, flattened or not).
data_files = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
scripts = glob.glob("/kaggle/input/**/train.py", recursive=True)
assert data_files, "Add Input: attach the private ClaimSense dataset"
assert scripts, "train.py not found in the dataset: upload the finetune folder too"
DATA = os.path.dirname(data_files[0])
TRAIN = scripts[0]
assert os.path.exists(os.path.join(os.path.dirname(TRAIN), "data.py")), "data.py missing"
print("data:", DATA)
print("script:", TRAIN)
print(sorted(os.listdir(DATA)))

# Smoke test first (a few minutes): SMOKE = False. Full training run: SMOKE = False.
SMOKE = False
EXTRA = "--limit 16 --epochs 1 --no-eval" if SMOKE else "--no-eval"
OUT = "/kaggle/working/claimsense-v2-e4b-smoke" if SMOKE else "/kaggle/working/claimsense-v2-e4b"


In [ ]:
!python {TRAIN} --data {DATA} --out {OUT} {EXTRA} --base-model unsloth/gemma-4-E4B-it

In [ ]:
import json
print(open(f'{OUT}/lengths.json').read())
log = json.load(open(f'{OUT}/training-log.json'))
print({k: v for k, v in log.items() if k != 'history'})
!ls -la {OUT}
!ls -la {OUT}/gguf 2>/dev/null || echo 'no GGUF yet (smoke run or export cell)'

In [ ]:
# GGUF export from the saved adapter, only when there is no GGUF yet: tests the
# export on the smoke adapter, and retries it if the full run's export failed.
import glob
!df -h /kaggle/working /tmp
if not glob.glob(f'{OUT}/gguf/*.gguf'):
    !python {TRAIN} --data {DATA} --out {OUT} --export-from {OUT}/adapter
!ls -la {OUT}/gguf